In [1]:
import numpy as np
import pandas as pd

# Load your cleaned tournament data
df = pd.read_csv('tourney_results_cleaned.csv')


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3410 entries, 0 to 3409
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Pod                3410 non-null   int64  
 1   Rank               3410 non-null   float64
 2   Name_Clean         3410 non-null   object 
 3   VP_Pile_1          3410 non-null   int64  
 4   VP_Pile_2          3410 non-null   int64  
 5   Total_VP           3410 non-null   int64  
 6   Win?               3410 non-null   object 
 7   Total Rank Points  3410 non-null   float64
 8   VP_Diff            3410 non-null   int64  
 9   Bystanders         3410 non-null   float64
 10  Fun_Level          3410 non-null   object 
 11  Win_Value          3410 non-null   object 
 12  Win_Points         3410 non-null   float64
 13  Turns_Value        3410 non-null   int64  
 14  Turns_Points       3410 non-null   float64
 15  Turns_Rank         3410 non-null   object 
 16  Escapes_Value      3410 

In [3]:
# 1. Convert Win_Value strings ('Yes'/'No') into numeric binary (1 or 0)
# We use .strip().lower() to protect against accidental spaces or mixed casing
df["Win_Numeric"] = df["Win_Value"].apply(
    lambda x: 1 if str(x).strip().lower() == "yes" else 0
)

# 2. Handle the special conditional rule for Total Turns Value
max_turns = df["Turns_Value"].max()


def calculate_turns_score(row):
    # Check our newly created numeric column for the win condition
    if row["Win_Numeric"] == 1:
        # Fewer turns is better for a Win: low turns yields a high performance value
        return max_turns - row["Turns_Value"]
    else:
        # More turns is better for a Loss: high turns yields a high performance value
        return row["Turns_Value"]


df["Adjusted_Turns_Value"] = df.apply(calculate_turns_score, axis=1)

print(
    "✅ Step 1 Complete: Mapped 'Yes'/'No' to 1/0 and preprocessed 'Adjusted_Turns_Value'."
)

✅ Step 1 Complete: Mapped 'Yes'/'No' to 1/0 and preprocessed 'Adjusted_Turns_Value'.


In [4]:
# Force all tournament scoring columns to be pure numeric floats
point_columns = ["Win_Points", "Escapes_Points", "Turns_Points", "PPT_Points"]
for col in point_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Calculate the maximum impact (Point Spread) each category can have on a match
point_spreads = {
    "Win_Rate_Weight": df["Win_Points"].max() - df["Win_Points"].min(),
    "Escapes_Weight": df["Escapes_Points"].max() - df["Escapes_Points"].min(),
    "Turns_Weight": df["Turns_Points"].max() - df["Turns_Points"].min(),
    "PPT_Weight": df["PPT_Points"].max() - df["PPT_Points"].min(),
}

total_spread = sum(point_spreads.values())

print("=== TOURNAMENT ENGINE DESIGN RATIOS ===")
print(
    f"Win Spread Impact:     {point_spreads['Win_Rate_Weight']} points max variance"
)
print(
    f"Escapes Spread Impact:   {point_spreads['Escapes_Weight']} points max variance"
)
print(
    f"Turns Spread Impact:     {point_spreads['Turns_Weight']} points max variance"
)
print(
    f"PPT Spread Impact:       {point_spreads['PPT_Weight']} points max variance"
)
print("-" * 40)

# Convert spreads directly into a percentage ratio of the total 100% weight pie
calculated_weights = {k: v / total_spread for k, v in point_spreads.items()}

print("=== FINAL CALCULATED MATHEMATICAL WEIGHTS ===")
for metric, weight in calculated_weights.items():
    print(f"{metric}: {weight:.4f} ({weight * 100:.1f}%)")

=== TOURNAMENT ENGINE DESIGN RATIOS ===
Win Spread Impact:     21.0 points max variance
Escapes Spread Impact:   19.5 points max variance
Turns Spread Impact:     27.0 points max variance
PPT Spread Impact:       27.0 points max variance
----------------------------------------
=== FINAL CALCULATED MATHEMATICAL WEIGHTS ===
Win_Rate_Weight: 0.2222 (22.2%)
Escapes_Weight: 0.2063 (20.6%)
Turns_Weight: 0.2857 (28.6%)
PPT_Weight: 0.2857 (28.6%)


In [5]:
# Map our Big Four raw value columns to their new normalized equivalents
metrics_to_normalize = {
    "Win_Numeric": "Win_Value_Normalized",
    "Escapes_Value": "Escapes_Value_Normalized",
    "Adjusted_Turns_Value": "Turns_Value_Normalized",
    "PPT_Value": "PPT_Value_Normalized",
}

for original_col, new_norm_col in metrics_to_normalize.items():
    col_min = df[original_col].min()
    col_max = df[original_col].max()

    if col_max - col_min > 0:
        df[new_norm_col] = (df[original_col] - col_min) / (col_max - col_min)
    else:
        df[new_norm_col] = 0.0

print(
    "✅ Step 3 Complete: Raw game values successfully scaled to 0-1 baselines."
)

✅ Step 3 Complete: Raw game values successfully scaled to 0-1 baselines.


In [6]:
# Compute the final weighted row score out of 100 maximum power points
# FIX: Updated the dictionary keys to match Cell 2 exactly!
df["Match_Power_Score"] = (
    (df["Win_Value_Normalized"] * calculated_weights["Win_Rate_Weight"])
    + (df["Escapes_Value_Normalized"] * calculated_weights["Escapes_Weight"])
    + (df["Turns_Value_Normalized"] * calculated_weights["Turns_Weight"])
    + (df["PPT_Value_Normalized"] * calculated_weights["PPT_Weight"])
) * 100

print(
    f"✅ Step 4 Complete: Match Power Scores successfully mapped across all {len(df)} rows."
)

✅ Step 4 Complete: Match Power Scores successfully mapped across all 3410 rows.


In [7]:
# 1. Split row metrics to isolate credit to Pick 1 and Pick 2 evenly
pick1_df = df[["Pick 1", "Match_Power_Score"]].rename(columns={"Pick 1": "Hero"})
pick2_df = df[["Pick 2", "Match_Power_Score"]].rename(columns={"Pick 2": "Hero"})

# 2. Stack data vertically and purge any unselected draft fields
combined_heroes_df = pd.concat([pick1_df, pick2_df], ignore_index=True).dropna(
    subset=["Hero"]
)

# 3. Calculate the raw performance average per hero
hero_performance = combined_heroes_df.groupby("Hero")["Match_Power_Score"].mean()

# 4. Calculate the total games played (sample size) for each hero
hero_game_counts = combined_heroes_df.groupby("Hero")["Match_Power_Score"].count()

# 5. CREATE THE CONFIDENCE MATRIX
# Find the global performance average (baseline) to pull low-sample heroes toward
global_mean_score = combined_heroes_df["Match_Power_Score"].mean()

# We set a threshold for full confidence (e.g., if a hero has played 15+ games, we fully trust them)
# You can change this number based on what your dataset's average games per hero looks like!
confidence_threshold = 15

# Calculate a confidence multiplier between 0.0 and 1.0 based on game volume
confidence_multipliers = hero_game_counts.apply(
    lambda x: min(x / confidence_threshold, 1.0)
)

# 6. Apply the confidence score to the raw performance
# Formula: (Raw Score * Confidence) + (Global Baseline * (1 - Confidence))
# This cleanly penalizes 1-game flukes without ruining their data
final_weighted_standings = (hero_performance * confidence_multipliers) + (
    global_mean_score * (1 - confidence_multipliers)
)

# Sort the final adjusted standings descending
final_leaderboard = final_weighted_standings.sort_values(ascending=False)

# Create a clean summary dataframe to print out nicely
summary_df = pd.DataFrame(
    {
        "Confidence-Adjusted Score": final_leaderboard,
        "Raw Avg Performance": hero_performance,
        "Games Played": hero_game_counts,
    }
).reindex(final_leaderboard.index)

print("\n=== THE DEFINITIVE CONFIDENCE-ADJUSTED HERO METAGAME RANKINGS ===")
print(summary_df.head(20).to_string(formatters={"Confidence-Adjusted Score": "{:.2f}".format, "Raw Avg Performance": "{:.2f}".format}))


=== THE DEFINITIVE CONFIDENCE-ADJUSTED HERO METAGAME RANKINGS ===
                        Confidence-Adjusted Score Raw Avg Performance  Games Played
Hero                                                                               
Spider-Gwen                                 51.41               51.41            31
Silk                                        49.38               49.38            25
Ultimate Spider-Man                         48.03               48.03            18
Peter's Allies                              47.03               49.36            10
Symbiote Spider-Man                         46.79               46.79            23
Ruby Summers                                46.01               47.34            11
Red Guardian                                45.78               45.78            27
Ant Army                                    45.77               45.77            15
Jessica Jones                               45.77               46.01            14
Gambit   

In [8]:
summary_df.head(20)

,Confidence-Adjusted Score,Raw Avg Performance,Games Played
Hero,,,
Spider-Gwen,51.408149,51.408149,31
Silk,49.375440,49.375440,25
Ultimate Spider-Man,48.027513,48.027513,18
Peter's Allies,47.025341,49.363244,10
Symbiote Spider-Man,46.793458,46.793458,23
Ruby Summers,46.008727,47.339343,11
Red Guardian,45.777479,45.777479,27
Ant Army,45.774113,45.774113,15
Jessica Jones,45.768345,46.012546,14
